# 14 — A1: nghiên cứu rò rỉ video trên laptop (PhoWhisper-small, 3 seed)

Hai bộ Train cùng 29.913 clip, chỉ khác nguồn gốc 722 clip. Chấm trên cùng tập `probe`: `probe_seen` (video có clip trong Train rò rỉ)
và `probe_unseen` (đối chứng). Kế hoạch và tiêu chí đọc kết quả: `reports/paper/PAPER_PLAN.md`.

- **Cấu hình đã đo trên máy này:** micro-batch 4 × tích lũy 8 (batch hiệu dụng vẫn 32), 8,5 giây/bước, GPU tổng 3,35 GB, RAM 1,6 GB.
- **Thời gian ước tính:** 2.802 bước ≈ 6,6 giờ train, thêm eval và giải mã probe khoảng 1 giờ: **gần 8 giờ mỗi lần chạy, khoảng 16 giờ mỗi cặp, 47 giờ cho cả 6 lần**.
- **Không trộn máy:** cả 6 lần chạy A1 phải cùng trên laptop. Đã chạy ở đây thì không chạy ô A1 trên Colab nữa.

Trước khi train: tắt kernel các notebook khác, đóng Chrome, ChatGPT, Zalo, cắm sạc, tắt sleep. Mỗi lần chỉ một job GPU.
Chạy lại ô train sẽ tiếp tục từ checkpoint gần nhất (lưu 10 phút một lần) và bỏ qua phần đã xong. Giữ nguyên `RUN_ID` và `DUTY` khi chạy lại.


In [1]:
from pathlib import Path
import sys, json, ctypes
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Mở notebook từ trong workspace dự án'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
assert PYTHON.is_file(), 'Chạy ASR/local/setup.ps1 trước'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'
RUNS = PROJECT/'local/runs'
RUN_ID, PROBE_ID, DUTY = 'local_a1', 'probe_a1', '1.0'
assert (PROJECT/'data/ViMedCSS_leak/probe.parquet').is_file(), 'Thiếu data/ViMedCSS_leak: chạy scripts/build_leak_data.py'

class _MemStatus(ctypes.Structure):
    _fields_ = [('dwLength', ctypes.c_ulong), ('dwMemoryLoad', ctypes.c_ulong), ('ullTotalPhys', ctypes.c_ulonglong),
                ('ullAvailPhys', ctypes.c_ulonglong), ('ullTotalPageFile', ctypes.c_ulonglong),
                ('ullAvailPageFile', ctypes.c_ulonglong), ('ullTotalVirtual', ctypes.c_ulonglong),
                ('ullAvailVirtual', ctypes.c_ulonglong), ('ullAvailExtendedVirtual', ctypes.c_ulonglong)]

def commit_free_gb():
    s = _MemStatus(); s.dwLength = ctypes.sizeof(s)
    ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(s))
    return s.ullAvailPageFile / 1e9

def gpu_job_running():
    """PID của job GPU local đang chạy (giữ local/gpu.lock), hoặc None."""
    import psutil
    lock = PROJECT/'local/gpu.lock'
    if not lock.exists():
        return None
    pid = int(lock.read_text())
    return pid if psutil.pid_exists(pid) else None

def adapter(exp):
    return RUNS/exp/RUN_ID/'final'

def probe_done(exp):
    return (RUNS/exp/PROBE_ID/'eval/probe/metrics.json').is_file()

def run_seeds(seeds):
    """Train rồi giải mã probe một lần cho từng thí nghiệm; phần đã xong được bỏ qua."""
    for seed in seeds:
        for cond in ['clean', 'leak']:
            exp = f'L_small_{cond}_s{seed}'
            if probe_done(exp):
                print('Đã xong, bỏ qua:', exp, flush=True)
                continue
            pid = gpu_job_running()
            if pid:
                print(f'Đang có job GPU khác (PID {pid}); dừng lại, không chạy thêm.')
                return
            if commit_free_gb() < 4:
                print(f'Bộ nhớ commit chỉ còn {commit_free_gb():.1f} GB; tắt bớt ứng dụng rồi chạy lại ô này.')
                return
            if not (adapter(exp)/'adapter_model.safetensors').is_file():
                run_cli([RUNNER, 'train', exp, '--run-id', RUN_ID, '--duty-cycle', DUTY])
            run_cli([RUNNER, 'eval', exp, '--run-id', PROBE_ID, '--split', 'probe', '--allow-test',
                     '--checkpoint', adapter(exp), '--duty-cycle', DUTY])
    print('Xong các seed', seeds)

print('Job GPU đang chạy:', gpu_job_running() or 'không')
free = commit_free_gb()
print(f'Bộ nhớ commit trống: {free:.1f} GB', '' if free >= 4 else '-> THẤP: đóng bớt ứng dụng rồi chạy lại ô này')
for s in [13, 14, 15]:
    for c in ['clean', 'leak']:
        print(f'L_small_{c}_s{s}:', 'xong' if probe_done(f'L_small_{c}_s{s}') else 'chưa xong')


Job GPU đang chạy: không
Bộ nhớ commit trống: 8.2 GB 
L_small_clean_s13: chưa xong
L_small_leak_s13: chưa xong
L_small_clean_s14: chưa xong
L_small_leak_s14: chưa xong
L_small_clean_s15: chưa xong
L_small_leak_s15: chưa xong


In [2]:
run_cli([RUNNER, 'check', 'L_small_leak_s13', '--run-id', RUN_ID])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" check L_small_leak_s13 --run-id local_a1
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261101_033037_697566.log
[INIT] mode=check experiment=L_small_leak_s13; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 0.7,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
  "system_ram_available_gb": 5.612703744,
  "gp

## 1. Seed 13 (cặp sạch và rò rỉ, khoảng 16 giờ)

Đây là ô cần chạy đầu tiên. Xong thì chạy ô **Xem kết quả** rồi gửi bảng đó.


In [3]:
run_seeds([13])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" train L_small_clean_s13 --run-id local_a1 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261101_033043_335556.log
[INIT] mode=train experiment=L_small_clean_s13; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
  "system_ram_available_gb":

## Theo dõi (chỉ đọc file, chạy được khi đang train ở ô khác)


In [4]:
def latest_state(run):
    ckpts = sorted((run/'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
    return json.loads((ckpts[-1]/'trainer_state.json').read_text(encoding='utf-8')) if ckpts else None

for s in [13, 14, 15]:
    for c in ['clean', 'leak']:
        exp = f'L_small_{c}_s{s}'
        st = latest_state(RUNS/exp/RUN_ID)
        if st:
            print(exp, f"bước {st['global_step']}/{st['max_steps']}",
                  {e['step']: round(e['eval_wer'], 4) for e in st['log_history'] if 'eval_wer' in e})


L_small_clean_s13 bước 2802/2802 {934: 0.1065, 1868: 0.1026, 2802: 0.1024}
L_small_leak_s13 bước 2802/2802 {934: 0.11, 1868: 0.1054, 2802: 0.1053}


## Xem kết quả

Chỉ dùng CPU. `did` là DiD, `did_lo` và `did_hi` là khoảng tin cậy 95% theo video. Nếu `did` của seed 13 không dương, được dừng ở đây và
báo là kết quả thử một seed (`PAPER_PLAN.md`). Nếu dương, chạy tiếp ô dưới: kết luận cần đủ 3 seed.


In [5]:
run_cli([PROJECT/'scripts/build_paper_tables.py', '--leak-only'])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\build_paper_tables.py" --leak-only
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261101_151445_831327.log
A balance
                    group  clips  hours  words_per_clip  words_per_sec  term_occ_per_clip  distinct_terms  same_text_in_clean  same_text_in_leaky  term_occ_in_clean_vocab  term_occ_in_leaky_vocab
    added to leaky TRAIN    722   1.41           26.14           3.72               1.05             210                 NaN                 NaN                      NaN                      NaN
dropped from leaky TRAIN    722   1.42           26.79           3.79               1.04             306                 NaN                 NaN                      NaN                      NaN
              probe_seen    809   1.51           24.78           3.69               1.04             213                25.0                31.0              

## 2. Seed 14 và 15 (khoảng 31 giờ)


In [6]:
run_seeds([14, 15])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" train L_small_clean_s14 --run-id local_a1 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261101_151449_764248.log
[INIT] mode=train experiment=L_small_clean_s14; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
  "system_ram_available_gb":

RuntimeError: Child process failed (1); read the error above. Log: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261101_151449_764248.log